# Stablecoin Pitch EDA: does new Tether issuance predict bitcoin?

**The mechanism.** USDT/USDC are minted when someone wires dollars in to buy crypto. New supply is cash arriving
on exchanges, which should be spent on bitcoin over the next days (a flow), versus minting simply following past
rallies (no prediction).

**Data.** DefiLlama daily circulating supply (`collect.py`), Binance BTC daily (UTC) closes. Signals: daily log
supply growth (bp) and its z-score vs the last 90 days. Outcomes: same day, next day, next 7 days. 2020–22 vs 2023–.

In [1]:
import pandas as pd, numpy as np, statsmodels.api as sm
s=pd.read_csv("data/stables.csv",parse_dates=["date"],index_col="date")
k=pd.read_csv("../Funding Pitch/data/klines_BTCUSDT.csv",parse_dates=["open_time"])
btc=k.set_index("open_time").close.resample("1D").last()          # UTC daily close (00:00 next day)
r=np.log(btc).diff()*1e4
D=pd.DataFrame({"r":r})
for c in ["total_usd","usdt_usd"]:
    g=np.log(s[c].replace(0,np.nan)).diff()*1e4                     # bp daily supply growth (DefiLlama date = 00:00 UTC snapshot)
    D[c]=g.reindex(D.index)
D["r_next"]=D.r.shift(-1); D["r_next7"]=D.r.shift(-1).rolling(7).sum().shift(-6)
D=D[D.index>="2020-01-01"].dropna()
D["z"]=(D.usdt_usd-D.usdt_usd.rolling(90).mean())/D.usdt_usd.rolling(90).std()
for per,m in [("2020-2022",D.index<"2023"),("2023-",D.index>="2023")]:
    d=D[m].dropna()
    for sig in ["usdt_usd","total_usd","z"]:
        for y_ in ["r","r_next","r_next7"]:
            f=sm.OLS(d[y_],sm.add_constant(d[sig])).fit(cov_type="HAC",cov_kwds={"maxlags":7})
            print(f"{per} {sig:9s} -> {y_:7s}: slope {f.params[sig]:+.3f} (t {f.tvalues[sig]:+.2f})")

2020-2022 usdt_usd  -> r      : slope +0.075 (t +0.90)
2020-2022 usdt_usd  -> r_next : slope +0.035 (t +0.79)
2020-2022 usdt_usd  -> r_next7: slope +0.251 (t +1.29)
2020-2022 total_usd -> r      : slope +0.067 (t +0.76)
2020-2022 total_usd -> r_next : slope +0.042 (t +0.88)
2020-2022 total_usd -> r_next7: slope +0.316 (t +1.21)
2020-2022 z         -> r      : slope +5.727 (t +0.32)
2020-2022 z         -> r_next : slope +1.375 (t +0.13)
2020-2022 z         -> r_next7: slope +23.942 (t +1.04)
2023- usdt_usd  -> r      : slope +1.046 (t +2.99)
2023- usdt_usd  -> r_next : slope +0.697 (t +1.99)
2023- usdt_usd  -> r_next7: slope +1.561 (t +1.54)
2023- total_usd -> r      : slope +0.447 (t +1.62)
2023- total_usd -> r_next : slope +0.071 (t +0.26)
2023- total_usd -> r_next7: slope +0.209 (t +0.22)
2023- z         -> r      : slope +24.901 (t +3.24)
2023- z         -> r_next : slope +19.224 (t +2.52)
2023- z         -> r_next7: slope +56.810 (t +2.52)


**Interpretation.**
- **2020–22: nothing** (all |t| ≤ 1.3).
- **2023–: USDT growth predicts BTC:** z-score → next day +19bp per SD (t 2.5), next 7 days +57bp (t 2.5), plus
  a strong same-day link (t 3.2). Total stablecoin supply does not predict (t 0.2–0.3). It is USDT-specific (offshore
  and exchange demand).
- **Regime-only again**, like the other crypto leads: no in-sample period where it held, and DefiLlama's snapshot time
  is approximate, so a same-day overlap cannot be ruled out.

**Verdict: weak lead.**